# Vision Transformer Object Detection - Kaggle Training Notebook

This notebook trains a Vision Transformer for object detection on foggy weather images using the PL-RT-DETR approach.

**Dataset**: VOC 2012 Filtered (with synthetic fog)

**Training Stages**:
1. **Teacher Network**: Train on clean images
2. **Student Network**: Train on foggy images with perceptual loss (knowledge distillation)

**Checkpoints**: Saved every 2 epochs to `/kaggle/working/` ⭐ (visible in file browser!)
**TensorBoard Logs**: `/kaggle/working/logs/`

## 1. Setup Kaggle Environment and Clone Repository

In [ ]:
import os
import sys

# Clone the repository from GitHub
!git clone https://github.com/habibour/fogg_object_det_vision_trans.git
os.chdir('/kaggle/working/fogg_object_det_vision_trans')

# To pull updates for already cloned repo, uncomment and run:
# !git pull origin main

print("✅ Repository cloned successfully!")
print(f"Current directory: {os.getcwd()}")
!ls -lh

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q ultralytics>=8.0.0
!pip install -q opencv-python-headless
!pip install -q albumentations
!pip install -q pycocotools
!pip install -q tensorboard

print("✅ All dependencies installed!")

# Verify installations
import torch
import ultralytics
import cv2
print(f"\n📦 Versions:")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
print(f"   Ultralytics: {ultralytics.__version__}")
print(f"   OpenCV: {cv2.__version__}")

## 3. Setup Dataset Paths (Kaggle Input)

In [ ]:
# Kaggle dataset paths
KAGGLE_VOC_PREFIX = "/kaggle/input/datasets/mdhabibourrahman/voc-2012-filtered"

# Dataset structure - point to VOC2012_paired directory
DATASET_ROOT = f"{KAGGLE_VOC_PREFIX}/voc_2012/processed/VOC2012_paired"
PAIRS_JSON = f"{DATASET_ROOT}/pairs.json"

# Working directories - save directly to /kaggle/working for UI visibility
OUTPUT_DIR = "/kaggle/working"
CHECKPOINT_DIR = OUTPUT_DIR  # ⭐ Save .pth files directly here (visible in UI)

# Create logs subdirectory for tensorboard
LOGS_DIR = f"{OUTPUT_DIR}/logs"
os.makedirs(LOGS_DIR, exist_ok=True)

print("📁 Dataset Paths:")
print(f"   Dataset Root: {DATASET_ROOT}")
print(f"   Pairs JSON: {PAIRS_JSON}")
print(f"   Output Dir: {OUTPUT_DIR}")
print(f"   Checkpoint Dir: {CHECKPOINT_DIR} ⭐ (.pth files visible in UI)")
print(f"   Tensorboard Logs: {LOGS_DIR}")

# Verify dataset exists
if os.path.exists(PAIRS_JSON):
    print("\n✅ Dataset found!")
    # Check dataset structure
    !ls -lh {DATASET_ROOT}/
    print("\n📂 ImageSets:")
    !ls -lh {DATASET_ROOT}/ImageSets/Main/ 2>/dev/null || echo "ImageSets not found"
else:
    print("\n⚠️  Dataset not found! Please add the dataset to Kaggle kernel.")
    print("   Expected path: /kaggle/input/datasets/mdhabibourrahman/voc-2012-filtered")

## 4. Configure Training Parameters

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================
# 
# Choose your training mode:
# 
# 🚀 QUICK TEST (Recommended for first run)
#    - teacher_epochs: 10
#    - student_epochs: 10
#    - Time: ~2-3 hours
#    - Purpose: Verify pipeline, check for errors
#    - Expected mAP: 0.60-0.75 (lower due to limited training)
#
# 🎯 FULL TRAINING (For paper-level results)
#    - teacher_epochs: 100
#    - student_epochs: 100
#    - Time: ~58 hours (on RTX 3090) or ~80+ hours (on T4/P100)
#    - Purpose: Reproduce paper results
#    - Expected mAP: 0.75-0.871 (baseline to paper-level)
#
# ⚙️ MONITORING:
#    - Loss should DECREASE steadily (not stay constant)
#    - If loss stuck ~0.1 or constant, check warnings in output
#    - Progress bar updates every 5% of batches
#
# 🔧 TUNING PERCEPTUAL WEIGHT:
#    - Default: 0.1 (balanced between detection and perceptual loss)
#    - If fog performance is low: increase to 0.2-0.3
#    - If clean performance drops: decrease to 0.05
#
# ============================================================================

config = {
    # Dataset
    'pairs_json': PAIRS_JSON,
    'dataset_root': DATASET_ROOT,
    
    # Model
    'model_name': 'rtdetr-l',  # RT-DETR Large
    'num_classes': 5,  # bicycle, bus, car, motorbike, person
    'img_size': 640,
    
    # Training
    'batch_size': 8,  # Reduce to 4 if OOM errors occur
    'num_workers': 2,
    'device': 'cuda',
    
    # ⚠️ CHOOSE YOUR TRAINING MODE (uncomment one set):
    
    # === QUICK TEST MODE (2-3 hours) ===
    'teacher_epochs': 10,
    'student_epochs': 10,
    
    # === FULL TRAINING MODE (58+ hours) === 
    # Uncomment below for paper-level results:
    # 'teacher_epochs': 100,
    # 'student_epochs': 100,
    
    # Optimization
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    
    # Loss weighting (tune based on results)
    # Increase if fog performance low, decrease if clean performance drops
    'perceptual_weight': 0.1,  # Default: 0.1 | Low fog perf: 0.2-0.3 | High clean perf: 0.05
    'perceptual_backbone': 'vgg19',  # Options: vgg16, vgg19, resnet50
    
    # Checkpointing
    'save_interval': 2,  # Save checkpoint every 2 epochs
    'val_interval': 10,  # Validate every 10 epochs
    
    # Paths
    'output_dir': OUTPUT_DIR,
    'checkpoint_dir': CHECKPOINT_DIR,
}

print("⚙️  Training Configuration:")
print(f"   Mode: {'QUICK TEST' if config['teacher_epochs'] <= 20 else 'FULL TRAINING'}")
print(f"   Total epochs: {config['teacher_epochs'] + config['student_epochs']}")
print(f"   Estimated time: {'~2-3 hours' if config['teacher_epochs'] <= 20 else '~58-80 hours'}")
print(f"\n📊 Key Settings:")
for key, value in config.items():
    if key in ['teacher_epochs', 'student_epochs', 'batch_size', 'perceptual_weight', 'learning_rate']:
        print(f"   {key}: {value}")
print("\n💡 Tip: Monitor loss - it should DECREASE steadily, not stay constant!")

## 5. Import Libraries and Load Modules

In [ ]:
# Add project root to path
sys.path.append('/kaggle/working/fogg_object_det_vision_trans')

# Import project modules
from dataset_loader import RTDETRDataset, collate_fn, create_dataloaders
from perceptual_loss import CombinedLoss, PerceptualLoss
from train_pl_rtdetr import PLRTDETRTrainer
from evaluate import Evaluator

# Standard imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from tqdm import tqdm
import json
from pathlib import Path
from datetime import datetime

# Ultralytics RT-DETR
from ultralytics import RTDETR

print("✅ All modules imported successfully!")
print(f"   Device: {torch.device(config['device'])}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 6. Load Dataset and Create DataLoaders

## 5.5. Create Train/Val Split Files (if missing)

In [ ]:
# Check if split files exist, if not create them from VOC2012_filtered or pairs.json
import shutil

imagesets_dir = f"{DATASET_ROOT}/ImageSets/Main"
os.makedirs(imagesets_dir, exist_ok=True)

# Check if train.txt and val.txt exist
train_file = os.path.join(imagesets_dir, 'train.txt')
val_file = os.path.join(imagesets_dir, 'val.txt')

if not os.path.exists(train_file) or not os.path.exists(val_file):
    print("⚠️  Split files missing, creating from VOC2012_filtered...")
    
    # Source directory for split files
    source_imagesets = f"{KAGGLE_VOC_PREFIX}/voc_2012/processed/VOC2012_filtered/ImageSets/Main"
    
    # Check if we can use VOC2012_filtered split files
    source_train = os.path.join(source_imagesets, 'train.txt')
    source_val = os.path.join(source_imagesets, 'val.txt')
    
    if os.path.exists(source_train):
        # Copy train.txt and val.txt from VOC2012_filtered
        print("   Copying split files from VOC2012_filtered...")
        shutil.copy(source_train, train_file)
        if os.path.exists(source_val):
            shutil.copy(source_val, val_file)
        else:
            # Create val.txt from test.txt if val doesn't exist
            source_test = os.path.join(source_imagesets, 'test.txt')
            if os.path.exists(source_test):
                shutil.copy(source_test, val_file)
        print("   ✅ Split files created!")
    else:
        # Create split files from pairs.json
        print("   Creating split files from pairs.json...")
        with open(PAIRS_JSON, 'r') as f:
            pairs_data = json.load(f)
        
        # Get all image IDs
        all_ids = [pair['id'] for pair in pairs_data['pairs']]
        
        # Split: 70% train, 15% val, 15% test
        np.random.seed(42)
        np.random.shuffle(all_ids)
        
        n_total = len(all_ids)
        n_train = int(0.70 * n_total)
        n_val = int(0.15 * n_total)
        
        train_ids = all_ids[:n_train]
        val_ids = all_ids[n_train:n_train+n_val]
        test_ids = all_ids[n_train+n_val:]
        
        # Write split files
        with open(train_file, 'w') as f:
            f.write('\n'.join(train_ids))
        
        with open(val_file, 'w') as f:
            f.write('\n'.join(val_ids))
        
        test_file = os.path.join(imagesets_dir, 'test.txt')
        with open(test_file, 'w') as f:
            f.write('\n'.join(test_ids))
        
        print(f"   ✅ Created splits: {len(train_ids)} train, {len(val_ids)} val, {len(test_ids)} test")
else:
    print("✅ Split files already exist!")

# Verify
print(f"\n📂 Split files:")
!ls -lh {imagesets_dir}/*.txt

In [ ]:
print("📦 Loading datasets...")

# Load pairs.json to check dataset info
with open(PAIRS_JSON, 'r') as f:
    pairs_data = json.load(f)

# Define classes (VOC filtered dataset)
CLASSES = ['bicycle', 'bus', 'car', 'motorbike', 'person']
    
print(f"\n📊 Dataset Statistics:")
print(f"   Total pairs: {len(pairs_data['pairs'])}")
print(f"   Fog levels: {pairs_data['metadata']['fog_levels']}")
print(f"   Classes: {CLASSES}")
print(f"   Number of classes: {len(CLASSES)}")

# Create datasets for Teacher training (clean images only)
train_dataset_clean = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='train',
    img_size=config['img_size'],
    use_foggy=False,
    return_both=False
)

val_dataset_clean = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='val',
    img_size=config['img_size'],
    use_foggy=False,
    return_both=False
)

# Create datasets for Student training (paired: both clean and foggy)
train_dataset_paired = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='train',
    img_size=config['img_size'],
    use_foggy=True,
    random_fog=True,
    return_both=True
)

val_dataset_foggy = RTDETRDataset(
    pairs_json_path=PAIRS_JSON,
    dataset_root=DATASET_ROOT,
    split='val',
    img_size=config['img_size'],
    use_foggy=True,
    fog_level='mid',
    return_both=True
)

# Create dataloaders
train_loader_clean = DataLoader(
    train_dataset_clean,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader_clean = DataLoader(
    val_dataset_clean,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

train_loader_paired = DataLoader(
    train_dataset_paired,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader_foggy = DataLoader(
    val_dataset_foggy,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True
)

print(f"\n✅ Dataloaders created:")
print(f"   Train (clean): {len(train_loader_clean)} batches")
print(f"   Val (clean): {len(val_loader_clean)} batches")
print(f"   Train (paired): {len(train_loader_paired)} batches")
print(f"   Val (foggy): {len(val_loader_foggy)} batches")

## 7. Stage 1: Train Teacher Network (Clean Images)

Train the teacher model on clean images from VOC 2012 dataset.

**⚠️ Note**: The current implementation uses placeholder loss values (0.5). For proper RT-DETR training with real loss computation, see `train_rtdetr_native.py` which uses Ultralytics' native `.train()` API. The current approach demonstrates the training pipeline structure but requires additional implementation for full loss calculation.

In [ ]:
print("="*60)
print("STAGE 1: Training Teacher Network on Clean Images")
print("="*60)

# Initialize RT-DETR teacher model
print("\n🤖 Loading RT-DETR model...")
teacher_model = RTDETR('rtdetr-l.pt')  # Load pretrained RT-DETR-L

# Configure for our 5 classes
# Note: We'll fine-tune the pretrained model for VOC filtered classes

# Setup tensorboard writer (save logs to /logs subdirectory)
writer = SummaryWriter(log_dir=f"{LOGS_DIR}/teacher_logs")

# Initialize teacher with config using the trainer class
print("\n🚀 Initializing teacher trainer...")
trainer = PLRTDETRTrainer(config)

# Train teacher
print(f"\n📚 Starting teacher training for {config['teacher_epochs']} epochs...")
print(f"   Checkpoints saved every {config['save_interval']} epochs to: {CHECKPOINT_DIR}")
print(f"   Validation runs every {config['val_interval']} epochs\n")

trainer.train_teacher()

print("\n✅ Teacher training completed!")
print(f"   Best model saved to: {CHECKPOINT_DIR}/teacher_best.pth")
print(f"   💡 .pth files are visible in the file browser!")

## 8. Stage 2: Train Student Network (Foggy Images with Perceptual Loss)

Train the student model on foggy images using knowledge distillation from the teacher.

In [ ]:
print("="*60)
print("STAGE 2: Training Student Network with Knowledge Distillation")
print("="*60)

# Load best teacher checkpoint
teacher_checkpoint_path = f"{CHECKPOINT_DIR}/teacher_best.pth"
if os.path.exists(teacher_checkpoint_path):
    print(f"\n✅ Loading best teacher model from: {teacher_checkpoint_path}")
else:
    print(f"\n⚠️  Teacher checkpoint not found at: {teacher_checkpoint_path}")
    print("   Using current teacher model state")

# Setup student tensorboard writer (save logs to /logs subdirectory)
student_writer = SummaryWriter(log_dir=f"{LOGS_DIR}/student_logs")

# Train student with perceptual loss
print(f"\n📚 Starting student training for {config['student_epochs']} epochs...")
print(f"   Using perceptual loss for knowledge distillation")
print(f"   Perceptual weight: {config['perceptual_weight']}")
print(f"   Checkpoints saved every {config['save_interval']} epochs to: {CHECKPOINT_DIR}")
print(f"   Validation runs every {config['val_interval']} epochs\n")

trainer.train_student()

print("\n✅ Student training completed!")
print(f"   Best model saved to: {CHECKPOINT_DIR}/student_best.pth")
print(f"   💡 .pth files are visible in the file browser!")

## 9. Evaluate Models on Test Set (Paper-Style Evaluation)

Comprehensive evaluation matching paper methodology:
- **Test Split**: Held-out test set (not used in training)
- **Conditions**: Clean, Fog Low (βF=0.005), Fog Mid (βF=0.01), Fog High (βF=0.02)
- **Metrics**: mAP@0.5, mAP@0.5:0.95, per-class AP
- **Comparison**: Teacher (baseline) vs Student (PL-RT-DETR)

**Paper Target Performance** (100 epochs):
- Clean: ~0.909 mAP
- Synthetic Fog (avg): ~0.871 mAP
- RTTS (real fog): ~0.422 mAP

In [ ]:
print("="*80)
print("PAPER-STYLE EVALUATION: Testing Models on Test Set")
print("="*80)

# Load best models
print("\n📥 Loading best models...")
teacher_best = f"{CHECKPOINT_DIR}/teacher_best.pth"
student_best = f"{CHECKPOINT_DIR}/student_best.pth"

# Verify test split exists
test_split_file = f"{DATASET_ROOT}/ImageSets/Main/test.txt"
if os.path.exists(test_split_file):
    print(f"✅ Test split found: {test_split_file}")
    test_split = 'test'
else:
    print(f"⚠️  Test split not found, using validation set")
    test_split = 'val'

# Create test datasets for all conditions (matching paper evaluation protocol)
print(f"\n📊 Creating test datasets (split: {test_split})...")

test_conditions = {
    'Clean': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split=test_split,
        img_size=config['img_size'],
        use_foggy=False,
        return_both=False
    ),
    'Fog_Low': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split=test_split,
        img_size=config['img_size'],
        use_foggy=True,
        fog_level='low',
        return_both=False
    ),
    'Fog_Mid': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split=test_split,
        img_size=config['img_size'],
        use_foggy=True,
        fog_level='mid',
        return_both=False
    ),
    'Fog_High': RTDETRDataset(
        pairs_json_path=PAIRS_JSON,
        dataset_root=DATASET_ROOT,
        split=test_split,
        img_size=config['img_size'],
        use_foggy=True,
        fog_level='high',
        return_both=False
    ),
}

print(f"   Conditions: {list(test_conditions.keys())}")
print(f"   Test samples per condition: {len(test_conditions['Clean'])}")

# Evaluate on each condition
results = {}
all_teacher_metrics = []
all_student_metrics = []

for condition_name, dataset in test_conditions.items():
    test_loader = DataLoader(
        dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=config['num_workers'],
        collate_fn=collate_fn,
        pin_memory=True
    )
    
    print(f"\n🔍 Evaluating on {condition_name}... (batches: {len(test_loader)})")
    
    # Initialize evaluators
    teacher_evaluator = Evaluator(trainer.teacher, device=config['device'])
    student_evaluator = Evaluator(trainer.student, device=config['device'])
    
    # Evaluate both models
    teacher_metrics = teacher_evaluator.evaluate_on_dataset(test_loader)
    student_metrics = student_evaluator.evaluate_on_dataset(test_loader)
    
    # Calculate improvement
    improvement = ((student_metrics['mAP'] - teacher_metrics['mAP']) / max(teacher_metrics['mAP'], 0.001)) * 100
    
    results[condition_name] = {
        'teacher': teacher_metrics,
        'student': student_metrics,
        'improvement_percent': improvement
    }
    
    all_teacher_metrics.append(teacher_metrics['mAP'])
    all_student_metrics.append(student_metrics['mAP'])

# Calculate average performance on fog conditions
fog_conditions = ['Fog_Low', 'Fog_Mid', 'Fog_High']
teacher_fog_avg = np.mean([results[c]['teacher']['mAP'] for c in fog_conditions])
student_fog_avg = np.mean([results[c]['student']['mAP'] for c in fog_conditions])
fog_improvement = ((student_fog_avg - teacher_fog_avg) / max(teacher_fog_avg, 0.001)) * 100

# Print paper-style table
print("\n" + "="*130)
print("EVALUATION RESULTS - PAPER FORMAT (Test Set)")
print("="*130)
print(f"\n{'Method':<25} {'Dataset':<12} {'Params':<10} {'Bicycle':<10} {'Bus':<10} {'Car':<10} {'Motorbike':<12} {'Person':<10} {'mAP(%)':<10}")
print("-" * 130)

# Helper function to get per-class AP (assuming evaluator returns class-wise metrics)
def get_class_aps(metrics):
    """Extract per-class AP values. Modify based on your Evaluator output."""
    # If evaluator returns per_class_AP, use it; otherwise use mAP as approximation
    if 'per_class_AP' in metrics:
        class_aps = metrics['per_class_AP']  # Should be list: [bicycle, bus, car, motorbike, person]
        return class_aps
    else:
        # Fallback: use mAP for all classes (you should update Evaluator to return per-class AP)
        return [metrics['mAP']] * 5

# Teacher (Baseline RT-DETR)
for condition in ['Clean', 'Fog_Low', 'Fog_Mid', 'Fog_High']:
    teacher_metrics = results[condition]['teacher']
    class_aps = get_class_aps(teacher_metrics)
    
    # Only show params for first row
    params_str = "71.3M" if condition == 'Clean' else ""
    method_name = f"RT-DETR (Teacher)" if condition == 'Clean' else ""
    
    print(f"{method_name:<25} {condition:<12} {params_str:<10} "
          f"{class_aps[0]*100:<10.2f} {class_aps[1]*100:<10.2f} {class_aps[2]*100:<10.2f} "
          f"{class_aps[3]*100:<12.2f} {class_aps[4]*100:<10.2f} {teacher_metrics['mAP']*100:<10.2f}")

print("-" * 130)

# Student (PL-RT-DETR)
for condition in ['Clean', 'Fog_Low', 'Fog_Mid', 'Fog_High']:
    student_metrics = results[condition]['student']
    class_aps = get_class_aps(student_metrics)
    
    # Only show params for first row
    params_str = "71.3M" if condition == 'Clean' else ""
    method_name = f"PL-RT-DETR (Student)" if condition == 'Clean' else ""
    
    print(f"{method_name:<25} {condition:<12} {params_str:<10} "
          f"{class_aps[0]*100:<10.2f} {class_aps[1]*100:<10.2f} {class_aps[2]*100:<10.2f} "
          f"{class_aps[3]*100:<12.2f} {class_aps[4]*100:<10.2f} {student_metrics['mAP']*100:<10.2f}")

print("=" * 130)

# Summary statistics
print(f"\n{'SUMMARY STATISTICS':<25} {'Clean':<12} {'Fog (Avg)':<12}")
print("-" * 50)
print(f"{'Teacher (RT-DETR)':<25} {results['Clean']['teacher']['mAP']*100:<12.2f} {teacher_fog_avg*100:<12.2f}")
print(f"{'Student (PL-RT-DETR)':<25} {results['Clean']['student']['mAP']*100:<12.2f} {student_fog_avg*100:<12.2f}")
print(f"{'Improvement (%)':<25} {results['Clean']['improvement_percent']:<12.2f} {fog_improvement:<12.2f}")
print("=" * 50)

# Paper targets comparison
print(f"\n{'PAPER TARGETS (100 epochs)':<30} {'Our Results ({} epochs)':<30}".format(config['teacher_epochs']))
print("-" * 60)
print(f"{'Clean Target: 90.9%':<30} {'Our Clean: {:.2f}%'.format(results['Clean']['student']['mAP']*100):<30}")
print(f"{'Fog Target: 87.1%':<30} {'Our Fog Avg: {:.2f}%'.format(student_fog_avg*100):<30}")
print("=" * 60)

print(f"\n💡 Note: Per-class AP breakdown requires Evaluator to return 'per_class_AP'.")
print(f"   Current implementation shows mAP. Update evaluate.py to compute per-class metrics.")

# Save detailed results
results['summary'] = {
    'clean_teacher': results['Clean']['teacher']['mAP'],
    'clean_student': results['Clean']['student']['mAP'],
    'fog_avg_teacher': teacher_fog_avg,
    'fog_avg_student': student_fog_avg,
    'fog_improvement_percent': fog_improvement,
    'test_split': test_split,
    'paper_targets': {
        'clean': 0.909,
        'synthetic_fog': 0.871,
        'rtts_real_fog': 0.422
    }
}

results_path = f"{OUTPUT_DIR}/evaluation_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Evaluation completed!")
print(f"   Table displayed above ☝️")
print(f"   Detailed results also saved to: {results_path}")
print(f"\n💡 Note: Lower performance than paper targets is expected for {config['teacher_epochs']}-epoch training.")
print(f"   For paper-level results, run 100-epoch training.")

## 10. Visualize Results (Paper-Style Comparison)

Generate publication-quality plots comparing:
- Teacher (baseline RT-DETR) vs Student (PL-RT-DETR)
- Current results vs Paper targets
- Performance breakdown by fog level

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (18, 12)

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Extract data (excluding 'summary' key)
eval_conditions = [k for k in results.keys() if k != 'summary']
teacher_maps = [results[c]['teacher']['mAP'] for c in eval_conditions]
student_maps = [results[c]['student']['mAP'] for c in eval_conditions]
improvements = [results[c]['improvement_percent'] for c in eval_conditions]

# Plot 1: Teacher vs Student mAP Comparison
ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(len(eval_conditions))
width = 0.35

bars1 = ax1.bar(x - width/2, teacher_maps, width, label='Teacher (RT-DETR)', 
                color='#4A90E2', edgecolor='black', linewidth=1.2)
bars2 = ax1.bar(x + width/2, student_maps, width, label='Student (PL-RT-DETR)', 
                color='#E85D75', edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax1.set_xlabel('Test Condition', fontsize=13, fontweight='bold')
ax1.set_ylabel('mAP@0.5', fontsize=13, fontweight='bold')
ax1.set_title('Model Performance Comparison (Test Set)', fontsize=15, fontweight='bold', pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(eval_conditions, fontsize=11)
ax1.legend(fontsize=11, loc='upper right')
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_ylim(0, max(max(teacher_maps), max(student_maps)) * 1.2)

# Plot 2: Improvement Percentage
ax2 = fig.add_subplot(gs[0, 1])
colors = ['#2ECC71' if imp > 0 else '#E74C3C' for imp in improvements]
bars = ax2.barh(eval_conditions, improvements, color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)

# Add value labels
for i, (bar, imp) in enumerate(zip(bars, improvements)):
    width = bar.get_width()
    label_x = width + (2 if width > 0 else -2)
    ax2.text(label_x, bar.get_y() + bar.get_height()/2., 
            f'{imp:+.1f}%', ha='left' if width > 0 else 'right', 
            va='center', fontsize=10, fontweight='bold')

ax2.set_xlabel('Improvement (%)', fontsize=13, fontweight='bold')
ax2.set_title('Student Improvement Over Teacher', fontsize=15, fontweight='bold', pad=15)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
ax2.grid(axis='x', alpha=0.3, linestyle='--')

# Plot 3: Paper Target Comparison
ax3 = fig.add_subplot(gs[1, 0])

# Data for paper comparison
categories = ['Clean', 'Fog\n(Average)']
paper_targets = [0.909, 0.871]
current_teacher = [results['summary']['clean_teacher'], results['summary']['fog_avg_teacher']]
current_student = [results['summary']['clean_student'], results['summary']['fog_avg_student']]

x_pos = np.arange(len(categories))
width = 0.25

bars1 = ax3.bar(x_pos - width, paper_targets, width, label='Paper Target (100 epochs)', 
                color='#95A5A6', edgecolor='black', linewidth=1.2, alpha=0.7)
bars2 = ax3.bar(x_pos, current_teacher, width, label=f'Teacher ({config["teacher_epochs"]} epochs)', 
                color='#4A90E2', edgecolor='black', linewidth=1.2)
bars3 = ax3.bar(x_pos + width, current_student, width, label=f'Student ({config["student_epochs"]} epochs)', 
                color='#E85D75', edgecolor='black', linewidth=1.2)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax3.set_ylabel('mAP@0.5', fontsize=13, fontweight='bold')
ax3.set_title('Performance vs Paper Targets', fontsize=15, fontweight='bold', pad=15)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(categories, fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3, linestyle='--')
ax3.set_ylim(0, 1.0)

# Plot 4: Fog Level Performance Breakdown
ax4 = fig.add_subplot(gs[1, 1])

fog_levels = ['Low\n(βF=0.005)', 'Mid\n(βF=0.01)', 'High\n(βF=0.02)']
fog_teacher = [results['Fog_Low']['teacher']['mAP'], 
               results['Fog_Mid']['teacher']['mAP'], 
               results['Fog_High']['teacher']['mAP']]
fog_student = [results['Fog_Low']['student']['mAP'], 
               results['Fog_Mid']['student']['mAP'], 
               results['Fog_High']['student']['mAP']]

x_fog = np.arange(len(fog_levels))
ax4.plot(x_fog, fog_teacher, 'o-', linewidth=3, markersize=10, 
         label='Teacher', color='#4A90E2', markeredgecolor='black', markeredgewidth=1.5)
ax4.plot(x_fog, fog_student, 's-', linewidth=3, markersize=10, 
         label='Student', color='#E85D75', markeredgecolor='black', markeredgewidth=1.5)

# Add value labels
for i, (t, s) in enumerate(zip(fog_teacher, fog_student)):
    ax4.text(i, t + 0.02, f'{t:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax4.text(i, s - 0.02, f'{s:.3f}', ha='center', fontsize=9, fontweight='bold', va='top')

ax4.set_xlabel('Fog Density Level', fontsize=13, fontweight='bold')
ax4.set_ylabel('mAP@0.5', fontsize=13, fontweight='bold')
ax4.set_title('Performance Degradation by Fog Density', fontsize=15, fontweight='bold', pad=15)
ax4.set_xticks(x_fog)
ax4.set_xticklabels(fog_levels, fontsize=11)
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3, linestyle='--')

# Overall title
fig.suptitle(f'PL-RT-DETR Evaluation Results - Test Set ({config["teacher_epochs"]} Training Epochs)', 
             fontsize=18, fontweight='bold', y=0.98)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/paper_style_evaluation.png", dpi=300, bbox_inches='tight')
plt.show()

print("="*80)
print("✅ Paper-style evaluation plots saved!")
print(f"   File: {OUTPUT_DIR}/paper_style_evaluation.png")
print("="*80)

In [ ]:
# Visualize sample predictions
print("\n📸 Visualizing sample predictions...")

# Load a few test samples
sample_loader = DataLoader(
    test_conditions['fog_mid'],
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# Get one batch
sample_batch = next(iter(sample_loader))
sample_images = sample_batch['images']
sample_targets = sample_batch['targets']

# Make predictions with student model
trainer.student.eval()
with torch.no_grad():
    predictions = trainer.student(sample_images.to(config['device']))

# Plot samples
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

class_names = ['bicycle', 'bus', 'car', 'motorbike', 'person']
colors = plt.cm.rainbow(np.linspace(0, 1, len(class_names)))

for idx in range(min(4, len(sample_images))):
    ax = axes[idx]
    
    # Convert image tensor to numpy
    img = sample_images[idx].permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1]
    
    ax.imshow(img)
    ax.set_title(f'Sample {idx + 1} - Mid Fog', fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # Note: Actual bounding box drawing would require parsing RT-DETR predictions
    # This is a placeholder - implement based on RT-DETR output format
    
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_predictions.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Sample predictions saved to: sample_predictions.png")

## 11. View Training Logs with TensorBoard

In [ ]:
# Load TensorBoard in Kaggle notebook
%load_ext tensorboard

# Launch TensorBoard
print("🔥 Launching TensorBoard...")
print(f"   Teacher logs: {LOGS_DIR}/teacher_logs")
print(f"   Student logs: {LOGS_DIR}/student_logs")

# View teacher training logs
%tensorboard --logdir {LOGS_DIR}/teacher_logs --port 6006

# To view student logs separately, uncomment:
# %tensorboard --logdir {LOGS_DIR}/student_logs --port 6007

## 12. List All Saved Checkpoints and Models

In [ ]:
print("="*60)
print("SAVED ARTIFACTS")
print("="*60)

# List all checkpoints in /kaggle/working
print("\n📦 Checkpoints (.pth files in /kaggle/working):")
!ls -lh {CHECKPOINT_DIR}/*.pth 2>/dev/null || echo "No .pth files yet"

print("\n📊 Output Files:")
!ls -lh {OUTPUT_DIR}/*.png {OUTPUT_DIR}/*.json 2>/dev/null || echo "No output files found"

print("\n📁 TensorBoard Logs:")
!ls -lh {LOGS_DIR}/

# Print summary of all saved files
import glob
from pathlib import Path

checkpoint_files = list(Path(CHECKPOINT_DIR).glob('*.pth'))
output_files = list(Path(OUTPUT_DIR).glob('*.png')) + list(Path(OUTPUT_DIR).glob('*.json'))

print(f"\n📌 Summary:")
print(f"   Total checkpoints: {len(checkpoint_files)}")
print(f"   Total output files: {len(output_files)}")
print(f"\n   💡 Checkpoint .pth files: {CHECKPOINT_DIR} ⭐ (visible in file browser)")
print(f"   📊 TensorBoard logs: {LOGS_DIR}")
print(f"   All files available for download from Kaggle")

# Create a summary file
summary = {
    'training_completed': datetime.now().isoformat(),
    'config': config,
    'evaluation_results': results,
    'checkpoints': [str(f.name) for f in checkpoint_files],
    'outputs': [str(f.name) for f in output_files]
}

summary_path = f"{OUTPUT_DIR}/training_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Training summary saved to: {summary_path}")

## 📝 Next Steps and Usage Instructions

### Download Trained Models
All checkpoints are saved in **`/kaggle/working/`** (visible in file browser!) and can be downloaded:
- **Best Teacher Model**: `teacher_best.pth` ⭐
- **Best Student Model**: `student_best.pth` ⭐
- **Intermediate Checkpoints**: `teacher_epoch_2.pth`, `teacher_epoch_4.pth`, etc.

TensorBoard logs are in `/kaggle/working/logs/` subdirectory.

### Evaluation Results
The notebook produces paper-style evaluation results:
- **Test Set Performance**: Teacher vs Student on all fog levels
- **Paper Comparison**: Current results vs published targets
- **Visualization**: Publication-quality plots saved to `paper_style_evaluation.png`
- **Detailed Metrics**: JSON file with complete results in `evaluation_results.json`

### To Use Trained Models:
```python
# Load student model for inference
from ultralytics import RTDETR
import torch

model = RTDETR('rtdetr-l.pt')
checkpoint = torch.load('/kaggle/working/student_best.pth')
model.model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Make predictions
results = model.predict('path/to/foggy/image.jpg')
```

### Update Code from GitHub:
To pull latest changes from the repository, run:
```python
!cd /kaggle/working/fogg_object_det_vision_trans && git pull origin main
```

### Resume Training:
To resume training from a checkpoint, modify the trainer initialization to load existing checkpoints.

### File Organization:
- 📦 **Checkpoints (.pth)**: `/kaggle/working/` ← **Visible in UI!**
- 📊 **TensorBoard logs**: `/kaggle/working/logs/` (event files)
- 🖼️ **Plots & Results**: `/kaggle/working/`
- 📄 **Evaluation JSON**: `/kaggle/working/evaluation_results.json`

### Key Results:
- Teacher model trained on clean images (baseline RT-DETR)
- Student model trained with perceptual loss (PL-RT-DETR)
- Checkpoints saved every 2 epochs
- Validation every 10 epochs
- Comprehensive test set evaluation matching paper methodology
- Performance comparison with paper targets (Clean: 0.909, Fog: 0.871)

### Expected Performance:
- **Quick Test (10 epochs)**: 60-75% of paper performance
- **Full Training (100 epochs)**: 75-100% of paper performance